In [0]:
# ML Validation Notebook – Clean Baseline cho risk_label

#Notebook này dùng để:
#- Kiểm tra khả năng label leakage do sử dụng các biến _flag.
#- Xây dựng baseline "clean" chỉ dùng các chỉ số đo (avg_hr, avg_sys, cholesterol, ...) và nhân khẩu học (age_est, gender, region).
#- So sánh kết quả với baseline ban đầu (có flags).


from pyspark.sql import functions as F
from pyspark.ml import Pipeline
from pyspark.ml.feature import (
    StringIndexer, OneHotEncoder, VectorAssembler, StandardScaler, IndexToString
)
from pyspark.ml.classification import LogisticRegression, RandomForestClassifier
from pyspark.ml.evaluation import MulticlassClassificationEvaluator


In [0]:
# Nếu cần, set lại catalog/schema giống lúc tạo bảng
spark.sql("USE CATALOG workspace")
spark.sql("USE SCHEMA health")

# 1. Load từ bảng Gold (đảm bảo giống notebook training chính)
df = spark.table("gold_mart_patient_day")

# (optional) filter nếu notebook chính có filter gì đặc biệt
# df = df.filter( ... )

df.printSchema()
df.select("risk_label").groupBy("risk_label").count().show()


In [0]:
# Giữ nguyên seed = 42 để kết quả nhất quán với notebook training
train_df, test_df = df.randomSplit([0.8, 0.2], seed=42)

print("Train:", train_df.count(), "rows")
print("Test :", test_df.count(), "rows")


In [0]:
label_col = "risk_label"

numeric_cols_with_flags = [
    "avg_hr", "avg_sys", "avg_dia", "avg_spo2",
    "cholesterol", "glucose", "hemoglobin", "age_est",
    "bp_flag", "spo2_flag", "glucose_flag", "chol_flag", "hemoglobin_flag", "total_flags"
]
categorical_cols = ["gender", "region"]

# Label indexer
label_indexer = StringIndexer(inputCol=label_col, outputCol="label", handleInvalid="keep")

cat_indexers = [
    StringIndexer(inputCol=c, outputCol=f"{c}_idx", handleInvalid="keep")
    for c in categorical_cols
]
cat_encoder = OneHotEncoder(
    inputCols=[f"{c}_idx" for c in categorical_cols],
    outputCols=[f"{c}_oh"  for c in categorical_cols],
    handleInvalid="keep"
)

feature_cols_with_flags = numeric_cols_with_flags + [f"{c}_oh" for c in categorical_cols]

assembler_with = VectorAssembler(
    inputCols=feature_cols_with_flags,
    outputCol="features_with"
)

rf_with = RandomForestClassifier(
    featuresCol="features_with",
    labelCol="label",
    numTrees=200,
    maxDepth=10,
    seed=42
)

pipe_with = Pipeline(stages=[label_indexer] + cat_indexers + [cat_encoder, assembler_with, rf_with])
rf_with_model = pipe_with.fit(train_df)
pred_with = rf_with_model.transform(test_df)

evaluator = MulticlassClassificationEvaluator(labelCol="label", predictionCol="prediction")
acc_with = evaluator.setMetricName("accuracy").evaluate(pred_with)
f1_with  = evaluator.setMetricName("f1").evaluate(pred_with)

print("Baseline WITH flags – accuracy:", acc_with, "  f1:", f1_with)


In [0]:
label_col = "risk_label"

numeric_cols_clean = [
    "avg_hr", "avg_sys", "avg_dia", "avg_spo2",
    "cholesterol", "glucose", "hemoglobin", "age_est"
]
categorical_cols = ["gender", "region"]

# Label indexer
label_indexer_clean = StringIndexer(inputCol=label_col, outputCol="label", handleInvalid="keep")

cat_indexers_clean = [
    StringIndexer(inputCol=c, outputCol=f"{c}_idx", handleInvalid="keep")
    for c in categorical_cols
]
cat_encoder_clean = OneHotEncoder(
    inputCols=[f"{c}_idx" for c in categorical_cols],
    outputCols=[f"{c}_oh"  for c in categorical_cols],
    handleInvalid="keep"
)

feature_cols_clean = numeric_cols_clean + [f"{c}_oh" for c in categorical_cols]

assembler_lr = VectorAssembler(inputCols=feature_cols_clean, outputCol="features_raw_lr")
scaler_lr    = StandardScaler(inputCol="features_raw_lr", outputCol="features_lr", withMean=True, withStd=True)

assembler_rf = VectorAssembler(inputCols=feature_cols_clean, outputCol="features_rf")


In [0]:
# Logistic Regression (multinomial)
lr = LogisticRegression(
    featuresCol="features_lr",
    labelCol="label",
    family="multinomial",
    maxIter=50,
    regParam=0.01,
    elasticNetParam=0.0
)

# Random Forest
rf = RandomForestClassifier(
    featuresCol="features_rf",
    labelCol="label",
    numTrees=200,
    maxDepth=10,
    seed=42
)

# Pipelines
pipe_lr_clean = Pipeline(stages=[label_indexer_clean] + cat_indexers_clean +
                                   [cat_encoder_clean, assembler_lr, scaler_lr, lr])

pipe_rf_clean = Pipeline(stages=[label_indexer_clean] + cat_indexers_clean +
                                   [cat_encoder_clean, assembler_rf, rf])

# Train
lr_clean_model = pipe_lr_clean.fit(train_df)
rf_clean_model = pipe_rf_clean.fit(train_df)

pred_lr_clean = lr_clean_model.transform(test_df)
pred_rf_clean = rf_clean_model.transform(test_df)


In [0]:
def eval_multiclass(pred_df, labelCol="label", predictionCol="prediction"):
    evalr = MulticlassClassificationEvaluator(labelCol=labelCol, predictionCol=predictionCol)
    metrics = {
        "accuracy": evalr.setMetricName("accuracy").evaluate(pred_df),
        "f1":       evalr.setMetricName("f1").evaluate(pred_df),
        "precision":evalr.setMetricName("weightedPrecision").evaluate(pred_df),
        "recall":   evalr.setMetricName("weightedRecall").evaluate(pred_df),
    }
    return metrics

metrics_lr_clean = eval_multiclass(pred_lr_clean)
metrics_rf_clean = eval_multiclass(pred_rf_clean)

print("=== CLEAN Baseline – Logistic Regression ===")
for k, v in metrics_lr_clean.items():
    print(f"{k}: {v:.4f}")

print("\n=== CLEAN Baseline – Random Forest ===")
for k, v in metrics_rf_clean.items():
    print(f"{k}: {v:.4f}")


In [0]:
# Lấy label mapping từ model
id2label = {i: lab for i, lab in enumerate(lr_clean_model.stages[0].labels)}

@F.udf("string")
def id_to_label(i):
    return id2label.get(int(i), "UNKNOWN")

def confusion_matrix(pred_df):
    cm = (pred_df
          .groupBy("label", "prediction")
          .count()
          .withColumn("label_txt", id_to_label("label"))
          .withColumn("pred_txt", id_to_label("prediction"))
          .select("label_txt", "pred_txt", "count"))
    cm_pivot = (cm.groupBy("label_txt")
                  .pivot("pred_txt")
                  .sum("count")
                  .fillna(0)
               )
    return cm_pivot.orderBy("label_txt")

print("=== Confusion Matrix – LR CLEAN ===")
display(confusion_matrix(pred_lr_clean))

print("=== Confusion Matrix – RF CLEAN ===")
display(confusion_matrix(pred_rf_clean))


In [0]:
rf_stage_clean = rf_clean_model.stages[-1]
importances = rf_stage_clean.featureImportances.toArray()

for name, imp in zip(feature_cols_clean, importances):
    print(f"{name}: {imp:.4f}")


In [0]:
from pyspark.ml.tuning import CrossValidator, ParamGridBuilder
from pyspark.ml.evaluation import MulticlassClassificationEvaluator

# Dùng lại rf và pipe_rf_clean nhưng sẽ để CrossValidator điều chỉnh tham số
rf_tune = RandomForestClassifier(
    featuresCol="features_rf",
    labelCol="label",
    seed=42
)

pipe_rf_tuned = Pipeline(stages=[label_indexer_clean] + cat_indexers_clean +
                                   [cat_encoder_clean, assembler_rf, rf_tune])

# Grid tham số không quá lớn để train kịp
paramGrid = (ParamGridBuilder()
    .addGrid(rf_tune.numTrees, [100, 200])
    .addGrid(rf_tune.maxDepth, [6, 10])
    .addGrid(rf_tune.minInfoGain, [0.0, 0.01])
    .build()
)

evaluator_cv = MulticlassClassificationEvaluator(
    labelCol="label",
    predictionCol="prediction",
    metricName="f1"
)

cv = CrossValidator(
    estimator=pipe_rf_tuned,
    estimatorParamMaps=paramGrid,
    evaluator=evaluator_cv,
    numFolds=3,
    parallelism=2,
    seed=42
)

cv_model_rf = cv.fit(train_df)
pred_rf_tuned = cv_model_rf.transform(test_df)

metrics_rf_tuned = eval_multiclass(pred_rf_tuned)

print("=== CLEAN RF (tuned with CV) ===")
for k, v in metrics_rf_tuned.items():
    print(f"{k}: {v:.4f}")

# Lấy bestModel để dùng sau này (vd: serving)
best_rf_model = cv_model_rf.bestModel
print("\nBest RF params:")
final_rf_stage = best_rf_model.stages[-1]
print("numTrees:", final_rf_stage.getNumTrees)
print("maxDepth:", final_rf_stage.getOrDefault("maxDepth"))
print("minInfoGain:", final_rf_stage.getOrDefault("minInfoGain"))


In [0]:
from pyspark.ml.classification import GBTClassifier

# GBT cho bài toán đa lớp: Spark thực chất internal là binary logistic per class one-vs-rest,
# nhưng bạn cứ dùng GBTClassifier trực tiếp.
gbt = GBTClassifier(
    featuresCol="features_rf",
    labelCol="label",
    maxDepth=6,
    maxIter=50,
    stepSize=0.1,
    seed=42
)

pipe_gbt = Pipeline(stages=[label_indexer_clean] + cat_indexers_clean +
                               [cat_encoder_clean, assembler_rf, gbt])

gbt_model = pipe_gbt.fit(train_df)
pred_gbt = gbt_model.transform(test_df)

metrics_gbt = eval_multiclass(pred_gbt)

print("=== CLEAN Baseline – GBTClassifier ===")
for k, v in metrics_gbt.items():
    print(f"{k}: {v:.4f}")

In [0]:
print("=== Confusion Matrix – GBT CLEAN ===")
display(confusion_matrix(pred_gbt))


In [0]:
# Giả sử bạn đã có:
# metrics_lr_clean, metrics_rf_clean, metrics_rf_tuned, metrics_gbt

from pyspark.sql import Row

rows = []

rows.append(Row(model="LogisticRegression_clean",
                accuracy=float(metrics_lr_clean["accuracy"]),
                f1=float(metrics_lr_clean["f1"]),
                precision=float(metrics_lr_clean["precision"]),
                recall=float(metrics_lr_clean["recall"])))

rows.append(Row(model="RandomForest_clean",
                accuracy=float(metrics_rf_clean["accuracy"]),
                f1=float(metrics_rf_clean["f1"]),
                precision=float(metrics_rf_clean["precision"]),
                recall=float(metrics_rf_clean["recall"])))

rows.append(Row(model="RandomForest_tuned_CV",
                accuracy=float(metrics_rf_tuned["accuracy"]),
                f1=float(metrics_rf_tuned["f1"]),
                precision=float(metrics_rf_tuned["precision"]),
                recall=float(metrics_rf_tuned["recall"])))

rows.append(Row(model="GBTClassifier_clean",
                accuracy=float(metrics_gbt["accuracy"]),
                f1=float(metrics_gbt["f1"]),
                precision=float(metrics_gbt["precision"]),
                recall=float(metrics_gbt["recall"])))

metrics_df = spark.createDataFrame(rows)
display(metrics_df.orderBy(F.desc("f1")))
